# MM-CAD config C: Tri-Modal + V (geometric vocabulary) injection
**Baseline (EmbeddingGemma + BRepFormer + DGCNN) + per-occurrence V descriptor + same-prototype attention bias.**

From-scratch, identical protocol to the baseline; the ONLY new parameters are `proto_proj` (43->512, 22,016) + 6 attn-bias scalars (+22,022, 0.08%). proto_data_v4.npz supplies the per-face proto_id + standardized 43-d descriptor.


In [ ]:
# Cell 1: Environment Setup
import os, sys
IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IS_COLAB:
    !pip install -q sentence-transformers plyfile h5py tensorboard psutil
    from google.colab import drive
    drive.mount('/content/drive')
else:
    # Local: ensure sentence-transformers, plyfile, h5py are installed
    pass

In [ ]:
# Cell: Setup CSV, H5, install lz4
# NOTE: PLY files are streamed directly from tar.lz4 into RAM (see data loading cell)
# to avoid disk space issues (122GB disk vs 176GB RAM on this Colab instance).
import os, shutil

if IS_COLAB:
    DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
    EXTRACT_DIR = '/content/mmcad_data'
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    # Install lz4 for streaming PLY extraction
    get_ipython().system('apt-get install -qq lz4 > /dev/null 2>&1')
    print('lz4 ready for streaming')

    # Copy CSV
    csv_src = f'{DRIVE_DIR}/abc_dataset_clean.csv'
    csv_dst = f'{EXTRACT_DIR}/abc_dataset_clean.csv'
    if not os.path.exists(csv_dst):
        shutil.copy2(csv_src, csv_dst)
        print(f'Copied CSV to {csv_dst}')
    else:
        print(f'CSV already at {csv_dst}')

    # Symlink H5 files from Drive (no disk usage)
    for h5_name in ['brep_features.h5', 'brep_topology.h5']:
        src = f'{DRIVE_DIR}/{h5_name}'
        dst = f'{EXTRACT_DIR}/{h5_name}'
        if os.path.exists(dst) or os.path.islink(dst):
            print(f'{h5_name} already at {dst}')
        elif os.path.exists(src):
            os.symlink(src, dst)
            print(f'Symlinked {h5_name}: {src} -> {dst}')
        else:
            print(f'{h5_name} not found on Drive')

    # Verify PLY archive exists
    PLY_TAR_LZ4 = f'{DRIVE_DIR}/ply.tar.lz4'
    if os.path.exists(PLY_TAR_LZ4):
        sz = os.path.getsize(PLY_TAR_LZ4) / 1e9
        print(f'PLY archive ready for streaming: {PLY_TAR_LZ4} ({sz:.1f} GB)')
    else:
        print(f'WARNING: PLY archive not found at {PLY_TAR_LZ4}')

    print('\nData setup complete.')
else:
    print('Local mode: assuming data files already in place.')



In [ ]:
# Cell 2: Imports
import os, sys, math, time, gc, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import h5py
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from plyfile import PlyData

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell: Configuration
class Config:
    # === PATHS ===
    if IS_COLAB:
        DATA_ROOT = '/content/mmcad_data'
        DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
        CSV_PATH = f'{DATA_ROOT}/abc_dataset_clean.csv'
        _local_h5 = f'{DATA_ROOT}/brep_features.h5'
        _drive_h5 = f'{DRIVE_DIR}/brep_features.h5'
        BREP_H5 = _local_h5 if os.path.exists(_local_h5) else _drive_h5
        _local_topo = f'{DATA_ROOT}/brep_topology.h5'
        _drive_topo = f'{DRIVE_DIR}/brep_topology.h5'
        BREP_TOPO_H5 = _local_topo if os.path.exists(_local_topo) else _drive_topo
        PLY_DIR = DATA_ROOT
        SAVE_DIR = f'{DRIVE_DIR}/baseline_trimodal_v4_C'
        LOCAL_SAVE_DIR = '/content/baseline_v4_C_ckpts'
    else:
        DATA_ROOT = r'C:\Users\anush\Desktop\MMCAD'
        CSV_PATH = os.path.join(DATA_ROOT, 'abc_dataset_clean.csv')
        BREP_H5 = os.path.join(DATA_ROOT, 'brep_features.h5')
        BREP_TOPO_H5 = os.path.join(DATA_ROOT, 'brep_topology.h5')
        PLY_DIR = DATA_ROOT
        SAVE_DIR = os.path.join(DATA_ROOT, 'baseline_trimodal_v4_C')
        LOCAL_SAVE_DIR = SAVE_DIR

    # === SHARED SPACE ===
    D_SHARED = 768
    MATRYOSHKA_DIMS = [128, 256, 512, 768]
    MATRYOSHKA_WEIGHTS = [1.0, 1.0, 1.0, 1.5]

    # === TEXT ENCODER ===
    TEXT_ENCODER_TYPE = 'embeddinggemma'
    TEXT_MODEL_GEMMA = 'google/embeddinggemma-300m'
    TEXT_MODEL_BERT = 'bert-base-uncased'
    TEXT_MAX_LENGTH = 512
    FREEZE_TEXT_AFTER_EPOCH = 6  # freeze starting epoch 6 (0-indexed)

    # === B-REP ENCODER (BRepFormer widened) ===
    FACE_DIM = 16
    D_BREP_MODEL = 512
    N_BREP_LAYERS = 6
    N_HEADS = 8
    MAX_FACES = 192
    MAX_EDGES = 512

    # === V (geometric vocabulary) injection — config C (the ONLY additions vs baseline) ===
    USE_PROTO_DESC = True          # add per-face 43-d V descriptor to face tokens
    USE_PROTO_ATTN_BIAS = True     # same-prototype attention bias (1 scalar/layer)
    PROTO_DATA = (f'{DRIVE_DIR}/vocab_v4/proto_data_v4.npz' if IS_COLAB
                  else os.path.join(DATA_ROOT, 'vocab_v4', 'proto_data_v4.npz'))

    # === PC ENCODER (DGCNN) ===
    NUM_POINTS = 2048
    DGCNN_K = 20
    DGCNN_LATENT = 1024

    # === POINT CLOUD AUGMENTATION (train only) ===
    PC_AUG_SCALE_JITTER = 0.05
    PC_AUG_NOISE_STD = 0.01

    # === TRAINING ===
    BATCH_SIZE = 64
    LEARNING_RATE = 1e-5
    BREP_LR_MULT = 10.0
    PC_LR_MULT = 10.0
    TEXT_WEIGHT_DECAY = 0.01
    THREE_D_WEIGHT_DECAY = 0.05
    NUM_EPOCHS = 12
    WARMUP_EPOCHS = 2
    GRAD_CLIP = 1.0

    # === MISC ===
    NUM_WORKERS = 4
    VAL_SPLIT = 0.1
    SEED = 42
    K_VALUES = [1, 5, 10]
    SAVE_EVERY = 5
    ENABLE_TENSORBOARD = True

config = Config()
os.makedirs(config.SAVE_DIR, exist_ok=True)
os.makedirs(config.LOCAL_SAVE_DIR, exist_ok=True)
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Save: {config.LOCAL_SAVE_DIR} (local) / {config.SAVE_DIR} (Drive)')
print(f'Text: {config.TEXT_ENCODER_TYPE}, freeze at epoch {config.FREEZE_TEXT_AFTER_EPOCH}')
print(f'BRep: BRepFormer d={config.D_BREP_MODEL}, L={config.N_BREP_LAYERS}')
print(f'PC:   DGCNN {config.NUM_POINTS}pts, k={config.DGCNN_K}')
print(f'Train: {config.NUM_EPOCHS} epochs, batch {config.BATCH_SIZE}')


In [ ]:
# Cell: DGCNN Point Cloud Encoder
# Source: Wang et al., "Dynamic Graph CNN" (TOG 2019)
# Standard 4 EdgeConv blocks. Proven stable across v1/v2 runs.

def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    return (-xx - inner - xx.transpose(2, 1)).topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20):
    bs, d, n = x.size()
    idx = knn(x, k)
    idx_base = torch.arange(0, bs, device=x.device).view(-1, 1, 1) * n
    idx = (idx + idx_base).view(-1)
    x = x.transpose(2, 1).contiguous()
    feat = x.view(bs * n, -1)[idx].view(bs, n, k, d)
    x = x.view(bs, n, 1, d).repeat(1, 1, k, 1)
    return torch.cat((feat - x, x), dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNEncoder(nn.Module):
    def __init__(self, latent_size=1024, k=20):
        super().__init__()
        self.k = k
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)
        self.bn5 = nn.BatchNorm1d(latent_size)
        self.bn6 = nn.BatchNorm1d(512)
        self.bn7 = nn.BatchNorm1d(latent_size)
        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, 1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, 1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, 1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, 1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(512, latent_size, 1, bias=False), self.bn5, nn.LeakyReLU(0.2))
        self.linear1 = nn.Linear(latent_size * 2, 512, bias=False)
        self.linear2 = nn.Linear(512, latent_size)
        self.dp = nn.Dropout(0.3)

    def forward(self, x):
        x1 = self.conv1(get_graph_feature(x, self.k)).max(dim=-1)[0]
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(dim=-1)[0]
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(dim=-1)[0]
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(dim=-1)[0]
        x = self.conv5(torch.cat((x1, x2, x3, x4), dim=1))
        x = torch.cat((x.max(2)[0], x.mean(2)), 1)
        x = self.dp(F.leaky_relu(self.bn6(self.linear1(x)), 0.2))
        return self.dp(self.bn7(self.linear2(x)))

print('DGCNN encoder defined (4 EdgeConv blocks, standard).')


In [ ]:
import gc
def cleanup_memory():
    gc.collect(); torch.cuda.empty_cache()
    if torch.cuda.is_available(): torch.cuda.synchronize()
    print(f"Memory cleaned. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
cleanup_memory()

In [ ]:
# Cell: BRepFormer Encoder — V-AUGMENTED (config C). Identical to baseline + two flag-gated
# additions: proto_proj (43->d_model) on face tokens, and a per-layer same-prototype attn bias.
#!/usr/bin/env python3
# =====================================================================
# v_brepformer.py  —  V-AUGMENTED BRepFormer (config C: per-occurrence descriptor + same-proto attn bias).
# ---------------------------------------------------------------------
# BYTE-FAITHFUL to the baseline BRepFormer (mmcad_baseline_trimodal_v4) EXCEPT two surgical additions,
# both gated by flags so the class is a drop-in:
#   (1) proto_proj: Linear(43, d_model) — adds each face's per-occurrence 43-d V descriptor to its token,
#       BEFORE the CLS concat and the transformer. Masked to feature faces (residual faces get zero shift).
#   (2) proto_attn_bias: one learned scalar per layer — added to attention logits between faces that share
#       a prototype id (residual/pad faces excluded; CLS excluded). Init 0 -> starts as a no-op.
# New params: 43*d_model + d_model (proj) + n_layers (scalars) = 22,534 at d_model=512, L=6 (+0.086%).
#
# proto_ids convention IN THE MODEL: residual / padding = -1 (the dataloader converts proto_data's M_star
# sentinel to -1). desc is zero on residual faces. Everything else is identical to the baseline encoder, so
# with both flags False (or proto_ids/proto_desc=None) the forward is bit-identical to the baseline.
# Offline shape/behaviour-tested by test_v_brepformer.py.
# =====================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return self.scale * x / torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.w1 = nn.Linear(d_in, d_hidden, bias=True)
        self.w2 = nn.Linear(d_hidden, d_in, bias=True)
        self.w3 = nn.Linear(d_in, d_hidden, bias=True)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class BRepFormerLayer(nn.Module):
    """Baseline layer + ONE new scalar (proto_attn_bias, init 0) added to same-prototype attention logits."""
    def __init__(self, d_model=256, n_heads=8, ffn_mult=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = SwiGLUFFN(d_model, d_model * ffn_mult)
        self.attn_drop = nn.Dropout(dropout)
        self.proto_attn_bias = nn.Parameter(torch.zeros(1))   # NEW: 1 scalar/layer, init 0 -> no-op at start

    def forward(self, x, attn_bias=None, mask=None, same_proto=None):
        B, N, D = x.shape
        h = self.norm1(x)
        q = self.q_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, N, self.n_heads, self.d_head).permute(0, 2, 1, 3)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if attn_bias is not None:
            attn = attn + attn_bias
        if same_proto is not None:                              # NEW: same-prototype bias (broadcasts over heads)
            attn = attn + self.proto_attn_bias * same_proto
        if mask is not None:
            attn = attn.masked_fill(mask[:, None, None, :] == 0, float('-inf'))
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        x = x + self.out_proj(out)
        x = x + self.ffn(self.norm2(x))
        return x


def compute_topo_distances(edge_to_faces, face_centroids, face_mask, face_normals, max_faces):
    """3 topological-distance channels for the attention bias (UNCHANGED from baseline)."""
    B = edge_to_faces.shape[0]
    N_f = max_faces
    device = edge_to_faces.device
    distances = torch.zeros(B, N_f, N_f, 3, device=device)
    for b in range(B):
        n_f = max(int(face_mask[b].sum().item()), 1)
        e2f = edge_to_faces[b].long()
        adj = [[] for _ in range(n_f)]
        for i in range(e2f.shape[0]):
            f0, f1 = e2f[i, 0].item(), e2f[i, 1].item()
            if 0 <= f0 < n_f and 0 <= f1 < n_f and f0 != f1:
                adj[f0].append(f1)
                adj[f1].append(f0)
        sp = torch.full((n_f, n_f), float('inf'), device=device)
        for src in range(n_f):
            sp[src, src] = 0
            queue, head = [src], 0
            while head < len(queue):
                u = queue[head]; head += 1
                for vv in adj[u]:
                    if sp[src, vv] == float('inf'):
                        sp[src, vv] = sp[src, u] + 1
                        queue.append(vv)
        finite = sp != float('inf')
        if finite.any():
            mx = sp[finite].max()
            sp[~finite] = mx + 1
            sp = sp / (mx + 1)
        distances[b, :n_f, :n_f, 0] = sp
        centroids = face_centroids[b, :n_f]
        cdist = torch.cdist(centroids.unsqueeze(0), centroids.unsqueeze(0)).squeeze(0)
        diag = (centroids.max(0).values - centroids.min(0).values).norm() + 1e-8
        distances[b, :n_f, :n_f, 1] = cdist / diag
        normals = face_normals[b, :n_f]
        n_norms = normals.norm(dim=1, keepdim=True).clamp(min=1e-8)
        normals_unit = normals / n_norms
        cos_sim = torch.mm(normals_unit, normals_unit.T).clamp(-1.0, 1.0)
        angular = torch.acos(cos_sim) / math.pi
        distances[b, :n_f, :n_f, 2] = angular
    return distances


class BRepFormerEncoder(nn.Module):
    """Baseline encoder + V additions (both flag-gated). proto_ids residual/pad = -1; desc 0 on residual."""
    def __init__(self, d_face_in=16, d_out=768, d_model=256, n_heads=8, n_layers=6, max_faces=192,
                 use_proto_desc=True, use_proto_attn_bias=True, proto_desc_dim=43):
        super().__init__()
        self.d_model = d_model
        self.max_faces = max_faces
        self.use_proto_desc = use_proto_desc
        self.use_proto_attn_bias = use_proto_attn_bias
        self.face_proj = nn.Linear(d_face_in, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.topo_bias_proj = nn.Sequential(
            nn.Linear(3, d_model), RMSNorm(d_model), nn.ReLU(), nn.Linear(d_model, n_heads))
        self.layers = nn.ModuleList([BRepFormerLayer(d_model, n_heads) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        # NEW: per-occurrence descriptor projection (the only new weight matrix)
        self.proto_proj = nn.Linear(proto_desc_dim, d_model)
        nn.init.normal_(self.proto_proj.weight, std=0.02)
        nn.init.zeros_(self.proto_proj.bias)

    def forward(self, face_features, face_centroids, edge_to_faces, face_mask, face_normals,
                proto_ids=None, proto_desc=None):
        B, N_f = face_features.shape[:2]
        x = self.face_proj(face_features)                       # (B, N_f, d_model)

        # (1) inject the per-occurrence V descriptor on feature faces (residual faces: proto_id < 0 -> no shift)
        if self.use_proto_desc and proto_desc is not None:
            real = (proto_ids >= 0).unsqueeze(-1).float() if proto_ids is not None else 1.0
            x = x + real * self.proto_proj(proto_desc)

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                          # (B, 1+N_f, d_model)

        topo_dist = compute_topo_distances(
            edge_to_faces, face_centroids, face_mask, face_normals, self.max_faces)
        topo_padded = F.pad(topo_dist, (0, 0, 1, 0, 1, 0), value=0)
        attn_bias = self.topo_bias_proj(topo_padded).permute(0, 3, 1, 2)

        # (2) same-prototype attention mask over [CLS, faces]; residual/pad (proto_id < 0) and CLS excluded
        same_proto = None
        if self.use_proto_attn_bias and proto_ids is not None:
            real = proto_ids >= 0                                # (B, N_f)
            sp = (proto_ids.unsqueeze(2) == proto_ids.unsqueeze(1)) & real.unsqueeze(2) & real.unsqueeze(1)
            sp = F.pad(sp, (1, 0, 1, 0), value=False)            # prepend CLS row/col (matches nobody)
            same_proto = sp.unsqueeze(1).to(x.dtype)             # (B, 1, 1+N_f, 1+N_f)

        mask = torch.cat([torch.ones(B, 1, device=face_features.device), face_mask], dim=1)
        for layer in self.layers:
            x = layer(x, attn_bias=attn_bias, mask=mask, same_proto=same_proto)

        return self.output_proj(self.final_norm(x)[:, 0, :])

print('V-augmented BRepFormer defined (config C: proto_proj + per-layer attn bias).')


In [ ]:
# Cell 6: Text Encoders
# Primary: EmbeddingGemma-300M (Matryoshka-trained, 768d native)
# Fallback: BERT-base-uncased (768d)

class EmbeddingGemmaEncoder(nn.Module):
    """Trainable EmbeddingGemma text encoder.
    Loads via sentence_transformers for the full pipeline,
    then runs forward manually to maintain gradient flow.
    Native Matryoshka dims: {128, 256, 512, 768}
    """
    def __init__(self, model_id='google/embeddinggemma-300m'):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        print(f'Loading EmbeddingGemma: {model_id}')
        st_model = SentenceTransformer(model_id, trust_remote_code=True)

        # Store all pipeline modules for gradient flow
        self._pipeline = nn.ModuleList(list(st_model))
        self.tokenizer = st_model.tokenizer

        for p in self.parameters():
            p.requires_grad = True

        n_params = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'  Pipeline: {len(self._pipeline)} modules, {n_params:.1f}M params')

    def forward(self, input_ids, attention_mask):
        features = {'input_ids': input_ids, 'attention_mask': attention_mask}
        for module in self._pipeline:
            features = module(features)
        return features['sentence_embedding']  # (B, 768), NOT normalized


class BertTextEncoder(nn.Module):
    """BERT-base text encoder (fallback if EmbeddingGemma unavailable)."""
    def __init__(self, model_name='bert-base-uncased', d_out=768):
        super().__init__()
        from transformers import BertModel, BertTokenizer
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)
        self.projection = nn.Linear(768, d_out)
        print(f'Loaded BERT: {model_name} -> {d_out}d')

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(out.last_hidden_state[:, 0, :])  # [CLS] token


def build_text_encoder(config):
    """Build text encoder based on config. Returns (encoder, tokenizer)."""
    if config.TEXT_ENCODER_TYPE == 'embeddinggemma':
        try:
            enc = EmbeddingGemmaEncoder(config.TEXT_MODEL_GEMMA)
            return enc, enc.tokenizer
        except Exception as e:
            print(f'EmbeddingGemma load failed: {e}')
            print('Falling back to BERT...')
            config.TEXT_ENCODER_TYPE = 'bert'

    enc = BertTextEncoder(config.TEXT_MODEL_BERT, d_out=config.D_SHARED)
    return enc, enc.tokenizer

print('Text encoders defined.')

In [ ]:
# Cell: Baseline Model + Matryoshka Loss

class DGCNNWithProjection(nn.Module):
    def __init__(self, d_out=768, k=20, latent_size=1024):
        super().__init__()
        self.dgcnn = DGCNNEncoder(latent_size=latent_size, k=k)
        self.projection = nn.Sequential(
            nn.Linear(latent_size, d_out),
            nn.LayerNorm(d_out),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_out, d_out),
        )
    def forward(self, x):
        return self.projection(self.dgcnn(x))


class Baseline(nn.Module):
    def __init__(self, text_encoder, config):
        super().__init__()
        self.text_encoder = text_encoder
        self.brep_encoder = BRepFormerEncoder(
            d_face_in=config.FACE_DIM, d_out=config.D_SHARED,
            d_model=config.D_BREP_MODEL, n_heads=config.N_HEADS,
            n_layers=config.N_BREP_LAYERS, max_faces=config.MAX_FACES,
            use_proto_desc=config.USE_PROTO_DESC, use_proto_attn_bias=config.USE_PROTO_ATTN_BIAS)
        self.pc_encoder = DGCNNWithProjection(
            d_out=config.D_SHARED, k=config.DGCNN_K, latent_size=config.DGCNN_LATENT)

        self.log_tau = nn.Parameter(torch.log(torch.tensor(0.07)))
        self.matryoshka_dims = config.MATRYOSHKA_DIMS

    @property
    def tau(self):
        return self.log_tau.exp().clamp(0.05, 1.0)

    def forward(self, batch):
        dev = next(self.parameters()).device
        z_text = self.text_encoder(
            batch['input_ids'].to(dev), batch['attention_mask'].to(dev))
        z_brep = self.brep_encoder(
            batch['face_features'].to(dev), batch['face_centroids'].to(dev),
            batch['edge_to_faces'].to(dev), batch['face_mask'].to(dev),
            batch['face_normals'].to(dev),
            proto_ids=batch['proto_ids'].to(dev), proto_desc=batch['proto_desc'].to(dev))
        z_pc = self.pc_encoder(batch['point_cloud'].to(dev))
        return {'z_text': z_text, 'z_brep': z_brep, 'z_pc': z_pc, 'tau': self.tau}


def info_nce_loss(z_a, z_b, temperature):
    z_a = F.normalize(z_a, dim=-1)
    z_b = F.normalize(z_b, dim=-1)
    B = z_a.shape[0]
    labels = torch.arange(B, device=z_a.device)
    logits_ab = torch.clamp(z_a @ z_b.T / temperature, -100, 100)
    logits_ba = torch.clamp(z_b @ z_a.T / temperature, -100, 100)
    loss = (F.cross_entropy(logits_ab, labels) + F.cross_entropy(logits_ba, labels)) / 2.0
    with torch.no_grad():
        pos = (F.normalize(z_a, dim=-1) * F.normalize(z_b, dim=-1)).sum(-1).mean()
        neg = (logits_ab * temperature).fill_diagonal_(0).sum() / max(B * (B - 1), 1)
        margin = pos - neg
    return loss, margin


def matryoshka_loss(outputs, dims, weights):
    z_t, z_b, z_p, tau = outputs['z_text'], outputs['z_brep'], outputs['z_pc'], outputs['tau']
    total = 0.0
    info = {}
    for d, w in zip(dims, weights):
        zt, zb, zp = z_t[:, :d], z_b[:, :d], z_p[:, :d]
        l_tb, m_tb = info_nce_loss(zt, zb, tau)
        l_tp, m_tp = info_nce_loss(zt, zp, tau)
        l_bp, m_bp = info_nce_loss(zb, zp, tau)
        scale_loss = (l_tb + l_tp + l_bp) / 3.0
        total += w * scale_loss
        info[f'loss_{d}d'] = scale_loss.item()
        info[f'margin_tb_{d}d'] = m_tb.item()
        info[f'margin_tp_{d}d'] = m_tp.item()
        info[f'margin_bp_{d}d'] = m_bp.item()
    return total / sum(weights), info

print('Baseline model and loss defined.')


In [ ]:
# Cell: Tri-Modal Dataset + Collate
# PC augmentation: fresh resample + scale jitter + gaussian noise (train only).
# No rotation (CAD models have canonical orientation).

def load_ply_fast(path):
    plydata = PlyData.read(path)
    return np.stack([plydata['vertex']['x'], plydata['vertex']['y'],
                     plydata['vertex']['z']], axis=1).astype(np.float32)


class TriModalDataset(Dataset):
    def __init__(self, df, config, compact_cache, topo_cache=None,
                 ply_cache=None, train_aug=False, proto_data=None):
        self.config = config
        self.ply_cache = {str(k).strip(): v for k, v in (ply_cache or {}).items()}
        self.train_aug = train_aug

        h5_uids = compact_cache['uids']
        self.uid_to_idx = {}
        for i, u in enumerate(h5_uids):
            k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
            self.uid_to_idx[k.strip()] = i

        self.face_features = compact_cache['face_features']
        self.face_masks = compact_cache['face_masks']
        self.edge_to_faces = compact_cache['edge_to_faces']
        self.face_centroids = compact_cache['face_centroids']

        # === V proto data (per-face proto_id + 43-d descriptor), brep-uid keyed ===
        self.proto_data = proto_data
        if proto_data is not None:
            self.proto_id = proto_data['proto_id']            # (N,192) int32 (M_star=residual/pad)
            self.proto_desc = proto_data['desc']              # (N,192,43) float32
            self.proto_M = int(proto_data['M_star'])
            self.proto_mean = proto_data['desc_mean'].astype(np.float32)
            self.proto_std = proto_data['desc_std'].astype(np.float32)
            self.proto_uid_to_idx = {str(u).strip(): i for i, u in enumerate(proto_data['uids'])}

        ff_flat = self.face_features.reshape(-1, self.face_features.shape[-1])
        valid = ff_flat[np.any(ff_flat != 0, axis=1)]
        self.ff_mean = valid.mean(0).astype(np.float32)
        self.ff_std = valid.std(0).astype(np.float32)
        self.ff_std[self.ff_std < 1e-6] = 1.0

        self.face_normals = None
        self.fn_uid_to_idx = {}
        if topo_cache and 'face_normals' in topo_cache:
            self.face_normals = topo_cache['face_normals']
            for i, u in enumerate(topo_cache['uids']):
                k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
                self.fn_uid_to_idx[k.strip()] = i

        df = df.copy()
        df['uid_str'] = df['uid'].astype(str).str.strip()

        # Filter breakdown for debugging
        in_brep = df['uid_str'].isin(self.uid_to_idx).sum()
        has_ply = (df['has_ply'] == True).sum()
        in_cache = df['uid_str'].isin(set(self.ply_cache.keys())).sum()

        valid_mask = (
            df['uid_str'].isin(self.uid_to_idx) &
            (df['has_ply'] == True) &
            df['uid_str'].isin(set(self.ply_cache.keys()))
        )
        self.samples = df[valid_mask].reset_index(drop=True)
        print(f'TriModalDataset ({"train+aug" if train_aug else "val"}): '
              f'{len(self.samples)} samples '
              f'(brep={in_brep}, has_ply={has_ply}, in_cache={in_cache})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        row = self.samples.iloc[i]
        uid_str = str(row['uid']).strip()
        idx = self.uid_to_idx[uid_str]
        n_faces = int(self.face_masks[idx].sum())

        # === BRep ===
        ff = (self.face_features[idx] - self.ff_mean) / self.ff_std
        ff[n_faces:] = 0
        fc = self.face_centroids[idx].copy()
        if n_faces > 0:
            center = fc[:n_faces].mean(0)
            scale = np.abs(fc[:n_faces] - center).max() + 1e-8
            fc = (fc - center) / scale
            fc[n_faces:] = 0
        if self.face_normals is not None and uid_str in self.fn_uid_to_idx:
            fn = self.face_normals[self.fn_uid_to_idx[uid_str]]
        else:
            fn = self.face_features[idx, :, 12:15]

        # === Point Cloud ===
        pc_full = self.ply_cache[uid_str]
        pts_idx = np.random.choice(
            len(pc_full), self.config.NUM_POINTS,
            replace=len(pc_full) < self.config.NUM_POINTS)
        pc_sampled = pc_full[pts_idx].copy()

        if self.train_aug:
            scale_f = 1.0 + (np.random.rand() - 0.5) * 2.0 * self.config.PC_AUG_SCALE_JITTER
            pc_sampled = pc_sampled * scale_f
            pc_sampled = pc_sampled + np.random.randn(*pc_sampled.shape).astype(np.float32) * self.config.PC_AUG_NOISE_STD

        pc_tensor = torch.from_numpy(pc_sampled).T.contiguous()

        # === Text ===
        title = str(row.get('title', ''))
        desc = str(row.get('description', ''))
        text = f'{title}. {desc}' if desc and desc != 'nan' else title

        # === V proto features (per face, brep order; standardized; residual/pad -> -1 / 0) ===
        MF = self.config.MAX_FACES
        if self.proto_data is not None and uid_str in self.proto_uid_to_idx:
            _pi = self.proto_uid_to_idx[uid_str]
            pr = self.proto_id[_pi].astype(np.int64)
            pr = np.where(pr == self.proto_M, -1, pr)            # model convention: residual/pad = -1
            pd = (self.proto_desc[_pi].astype(np.float32) - self.proto_mean) / self.proto_std
            pd = np.clip(pd, -8.0, 8.0); pd[pr < 0] = 0.0        # no descriptor on residual faces
            proto_ids = torch.from_numpy(pr)
            proto_desc = torch.from_numpy(pd)
        else:
            proto_ids = torch.full((MF,), -1, dtype=torch.long)
            proto_desc = torch.zeros((MF, 43), dtype=torch.float32)

        return {
            'face_features': torch.tensor(ff, dtype=torch.float32),
            'face_centroids': torch.tensor(fc, dtype=torch.float32),
            'edge_to_faces': torch.tensor(self.edge_to_faces[idx], dtype=torch.long),
            'face_mask': torch.tensor(self.face_masks[idx], dtype=torch.float32),
            'face_normals': torch.tensor(fn, dtype=torch.float32),
            'point_cloud': pc_tensor,
            'text': text,
            'proto_ids': proto_ids,
            'proto_desc': proto_desc,
            'uid': uid_str,
        }


def make_collate_fn(tokenizer, max_length=512, use_task_prompt=True):
    def collate_fn(batch):
        texts = [item['text'] for item in batch]
        if use_task_prompt:
            texts = [f'title: none | text: {t}' for t in texts]
        tokens = tokenizer(texts, padding=True, truncation=True,
                           max_length=max_length, return_tensors='pt')
        return {
            'input_ids': tokens['input_ids'],
            'attention_mask': tokens['attention_mask'],
            'face_features': torch.stack([b['face_features'] for b in batch]),
            'face_centroids': torch.stack([b['face_centroids'] for b in batch]),
            'edge_to_faces': torch.stack([b['edge_to_faces'] for b in batch]),
            'face_mask': torch.stack([b['face_mask'] for b in batch]),
            'face_normals': torch.stack([b['face_normals'] for b in batch]),
            'point_cloud': torch.stack([b['point_cloud'] for b in batch]),
            'proto_ids': torch.stack([b['proto_ids'] for b in batch]),
            'proto_desc': torch.stack([b['proto_desc'] for b in batch]),
            'uids': [b['uid'] for b in batch],
        }
    return collate_fn

print('Dataset and collate defined (with train-time PC augmentation).')


In [ ]:
# Cell: Training, Validation, Evaluation Functions

def train_epoch(model, loader, optimizer, scheduler, config):
    model.train()
    text_frozen = not any(p.requires_grad for p in model.text_encoder.parameters())
    if text_frozen:
        model.text_encoder.eval()
    total_loss = 0
    n = 0
    margin_accum = {}
    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch)
        loss, info = matryoshka_loss(outputs, config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS)
        if not torch.isfinite(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], config.GRAD_CLIP)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        n += 1
        for k, v in info.items():
            margin_accum[k] = margin_accum.get(k, 0) + v
        if n % 20 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'tau': f'{outputs["tau"].item():.4f}'})
    avg_info = {k: v / max(n, 1) for k, v in margin_accum.items()}
    return total_loss / max(n, 1), avg_info


@torch.no_grad()
def validate_epoch(model, loader, config):
    model.eval()
    total_loss = 0
    n = 0
    margin_accum = {}
    for batch in tqdm(loader, desc='Val', leave=False):
        outputs = model(batch)
        loss, info = matryoshka_loss(outputs, config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS)
        total_loss += loss.item()
        n += 1
        for k, v in info.items():
            margin_accum[k] = margin_accum.get(k, 0) + v
    avg_info = {k: v / max(n, 1) for k, v in margin_accum.items()}
    return total_loss / max(n, 1), avg_info


@torch.no_grad()
def evaluate_matryoshka(model, loader, config):
    """Fast GPU eval: vectorized rank computation, no numpy loops."""
    import gc
    model.eval()
    all_zt, all_zb, all_zp = [], [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        outputs = model(batch)
        all_zt.append(outputs['z_text'].float())
        all_zb.append(outputs['z_brep'].float())
        all_zp.append(outputs['z_pc'].float())
        del outputs

    z_text, z_brep, z_pc = torch.cat(all_zt), torch.cat(all_zb), torch.cat(all_zp)
    del all_zt, all_zb, all_zp
    N = z_text.shape[0]
    results = {}

    for dim in config.MATRYOSHKA_DIMS:
        zt = F.normalize(z_text[:, :dim], dim=-1)
        zb = F.normalize(z_brep[:, :dim], dim=-1)
        zp = F.normalize(z_pc[:, :dim], dim=-1)
        for name, q, g in [
            ('text2brep', zt, zb), ('brep2text', zb, zt),
            ('text2pc', zt, zp), ('pc2text', zp, zt),
            ('brep2pc', zb, zp), ('pc2brep', zp, zb),
        ]:
            sim = q @ g.T
            gt = torch.arange(N, device=sim.device)
            sorted_idx = sim.argsort(dim=-1, descending=True)
            ranks = (sorted_idx == gt.unsqueeze(-1)).float().argmax(dim=-1)
            del sim, sorted_idx
            for k in config.K_VALUES:
                hit = (ranks < k).float().mean().item() * 100
                ap = torch.where(ranks < k, 1.0/(ranks.float()+1), torch.zeros_like(ranks, dtype=torch.float32)).mean().item() * 100
                results[f'{name}_recall@{k}_{dim}d'] = hit
                results[f'{name}_mAP@{k}_{dim}d'] = ap
            del ranks
        del zt, zb, zp
    del z_text, z_brep, z_pc
    gc.collect(); torch.cuda.empty_cache()
    return results


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, save_dir, name, light=False):
    path = os.path.join(save_dir, name)
    ckpt = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'metrics': metrics}
    if not light:
        ckpt['optimizer_state_dict'] = optimizer.state_dict()
        ckpt['scheduler_state_dict'] = scheduler.state_dict() if scheduler else None
    torch.save(ckpt, path)

def sync_to_drive(local_path, drive_path):
    if local_path == drive_path: return
    import shutil
    try:
        os.makedirs(os.path.dirname(drive_path), exist_ok=True)
        shutil.copy2(local_path, drive_path)
    except Exception as e:
        print(f'  [warn] sync failed: {e}')

print('Train/eval functions defined (fast GPU eval).')


In [ ]:
# Cell: Load Data
# Fast path: load preprocessed abc_point_clouds.h5 if present on Drive
# Slow path: stream from ply.tar.lz4 (takes ~2 hours on Colab)
import psutil
import time

print('=' * 60)
print('LOADING DATA')
print('=' * 60)

# === CSV ===
df = pd.read_csv(config.CSV_PATH)
print(f'CSV: {len(df)} rows')

# === Compact BRep H5 ===
_compact_cache = {}
with h5py.File(config.BREP_H5, 'r') as f:
    _compact_cache['face_features'] = f['face_features'][:]
    _compact_cache['face_masks'] = f['face_masks'][:]
    _compact_cache['edge_to_faces'] = f['edge_to_faces'][:]
    _compact_cache['face_centroids'] = f['face_centroids'][:]
    _compact_cache['uids'] = f['uids'][:]
total_gb = sum(v.nbytes for v in _compact_cache.values() if hasattr(v, 'nbytes')) / 1e9
print(f'Compact BRep: {total_gb:.1f} GB ({len(_compact_cache["uids"])} samples)')

# === Topology H5 ===
_topo_cache = None
topo_path = config.BREP_TOPO_H5
if os.path.exists(topo_path):
    _topo_cache = {}
    with h5py.File(topo_path, 'r') as f:
        if 'face_normals' in f:
            _topo_cache['face_normals'] = f['face_normals'][:]
        _topo_cache['uids'] = f['uids'][:]
    topo_gb = sum(v.nbytes for v in _topo_cache.values() if hasattr(v, 'nbytes')) / 1e9
    print(f'Topology H5: {topo_gb:.1f} GB (face_normals loaded)')
else:
    print(f'Topology H5 not found. Using face_features[:, 12:15] fallback.')

# === PLY data ===
# Try preprocessed H5 first (fast path)
_ply_cache = {}
_ply_presampled = None  # (N, num_points, 3) array
_ply_uid_to_idx = {}

PREPROCESSED_PLY_H5 = f'{config.DRIVE_DIR}/abc_point_clouds.h5' if IS_COLAB else os.path.join(config.DATA_ROOT, 'abc_point_clouds.h5')

if os.path.exists(PREPROCESSED_PLY_H5):
    print(f'\nFound preprocessed PLY H5: {PREPROCESSED_PLY_H5}')
    print('Loading into RAM (fast path)...')
    start = time.time()
    with h5py.File(PREPROCESSED_PLY_H5, 'r') as f:
        _ply_presampled = f['point_clouds'][:]  # (N, num_points, 3)
        ply_uids_raw = f['uids'][:]
        num_preset = int(f.attrs.get('num_points', _ply_presampled.shape[1]))
    for i, u in enumerate(ply_uids_raw):
        k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
        _ply_uid_to_idx[k.strip()] = i
    ply_gb = _ply_presampled.nbytes / 1e9
    print(f'Loaded {len(_ply_uid_to_idx)} PLYs in {time.time()-start:.1f}s ({ply_gb:.1f} GB)')
    print(f'Pre-sampled to {num_preset} points per model')
else:
    print(f'\nPreprocessed PLY H5 not found at {PREPROCESSED_PLY_H5}')
    print('Falling back to tar.lz4 streaming (slow, ~2 hours)...')
    import subprocess, tarfile, io

    # Build target UID set
    brep_uids = set()
    for u in _compact_cache['uids']:
        k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
        brep_uids.add(k.strip())
    csv_ply_uids = set()
    for _, row in df.iterrows():
        if row.get('has_ply', False):
            uid = str(row['uid']).strip()
            if uid in brep_uids:
                csv_ply_uids.add(uid)
    print(f'Target PLY UIDs: {len(csv_ply_uids)}')

    PLY_TAR_LZ4 = f'{config.DRIVE_DIR}/ply.tar.lz4' if IS_COLAB else os.path.join(config.DATA_ROOT, 'ply.tar.lz4')
    n_ok = n_fail = n_skip = 0
    start = time.time()
    proc = subprocess.Popen(['lz4', '-dc', PLY_TAR_LZ4], stdout=subprocess.PIPE, bufsize=1024*1024*16)
    try:
        tar = tarfile.open(fileobj=proc.stdout, mode='r|')
        pbar = tqdm(desc='Streaming PLY', total=len(csv_ply_uids))
        for member in tar:
            if not member.isfile() or not member.name.endswith('.ply'):
                continue
            uid = os.path.basename(member.name).replace('.ply', '').strip()
            if uid not in csv_ply_uids:
                n_skip += 1
                tar.members = []
                continue
            f_ply = tar.extractfile(member)
            if f_ply is None:
                n_fail += 1
                continue
            try:
                data = f_ply.read()
                plydata = PlyData.read(io.BytesIO(data))
                xyz = np.stack([plydata['vertex']['x'], plydata['vertex']['y'],
                                plydata['vertex']['z']], axis=1).astype(np.float32)
                xyz = xyz - xyz.mean(0)
                norm = np.max(np.linalg.norm(xyz, axis=1))
                if norm > 0:
                    xyz = xyz / norm
                _ply_cache[uid] = xyz
                n_ok += 1
                pbar.update(1)
            except Exception:
                n_fail += 1
            tar.members = []
        pbar.close()
        tar.close()
    finally:
        proc.stdout.close()
        proc.wait()
    print(f'\nStreamed {n_ok} in {time.time()-start:.0f}s ({n_fail} failed, {n_skip} skipped)')

mem = psutil.virtual_memory()
print(f'\nRAM: {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB ({mem.percent}%)')
print('=' * 60)

# === V proto data (per-face proto_id + standardized descriptor) ===
_proto_data = None
if os.path.exists(config.PROTO_DATA):
    _pz = np.load(config.PROTO_DATA, allow_pickle=True)
    _proto_data = {k: _pz[k] for k in _pz.files}
    _M = int(_proto_data['M_star'])
    print(f'V proto_data: proto_id {_proto_data["proto_id"].shape} | M*={_M} | '
          f'feature faces {int((_proto_data["proto_id"] != _M).sum()):,}')
else:
    print(f'WARNING: proto_data NOT found at {config.PROTO_DATA} -> config-C V injection will be all-residual!')


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Force-rebuild _ply_cache directly from the preprocessed H5 file
import h5py, gc, time

H5_PATH = '/content/drive/MyDrive/MMCAD/abc_point_clouds.h5'

# Wipe anything stale
try:
    del _ply_cache
except NameError:
    pass
gc.collect()

print(f'Loading {H5_PATH} ...')
t0 = time.time()

with h5py.File(H5_PATH, 'r') as f:
    print(f'  H5 keys: {list(f.keys())}')
    for k in f.keys():
        ds = f[k]
        print(f'    {k}: shape={ds.shape}, dtype={ds.dtype}')

    # Identify the arrays (names may vary)
    uid_key = next((k for k in f.keys() if 'uid' in k.lower()), None)
    pts_key = next((k for k in f.keys() if k != uid_key), None)
    print(f'  Using uid_key={uid_key!r}, pts_key={pts_key!r}')

    uids = f[uid_key][:]
    points = f[pts_key][:]

_ply_cache = {}
for i, u in enumerate(uids):
    k = u.decode('utf-8') if isinstance(u, bytes) else str(u)
    _ply_cache[k.strip()] = points[i]

del uids, points
gc.collect()

print(f'\n_ply_cache: {len(_ply_cache)} entries ({time.time()-t0:.0f}s)')
print(f'Sample keys: {list(_ply_cache.keys())[:5]}')
sample_pc = _ply_cache[list(_ply_cache.keys())[0]]
print(f'Sample shape: {sample_pc.shape}, dtype: {sample_pc.dtype}')

# Verify overlap
csv_uids = set(df['uid'].astype(str).str.strip())
brep_uids = {u.decode() if isinstance(u, bytes) else str(u) for u in _compact_cache['uids']}
brep_uids = {u.strip() for u in brep_uids}
overlap = len(csv_uids & brep_uids & set(_ply_cache.keys()))
print(f'\nCSV ∩ BRep ∩ PLY overlap: {overlap}')


In [ ]:
# Cell 11: Build Text Encoder + Tokenizer

text_encoder, tokenizer = build_text_encoder(config)
print(f'\nText encoder: {config.TEXT_ENCODER_TYPE}')

# Determine if we should use task prompts (EmbeddingGemma yes, BERT no)
use_task_prompt = (config.TEXT_ENCODER_TYPE == 'embeddinggemma')

In [ ]:
# Cell: Create Datasets + DataLoaders

df_valid = df[df['has_ply'] == True].copy()
df_valid['uid_str'] = df_valid['uid'].astype(str).str.strip()
df_valid = df_valid[df_valid['uid_str'].isin(_ply_cache)].reset_index(drop=True)

n_val = int(len(df_valid) * config.VAL_SPLIT)
indices = np.random.RandomState(config.SEED).permutation(len(df_valid))
df_train = df_valid.iloc[indices[n_val:]].reset_index(drop=True)
df_val = df_valid.iloc[indices[:n_val]].reset_index(drop=True)
print(f'Train: {len(df_train)}, Val: {len(df_val)}')

train_dataset = TriModalDataset(
    df_train, config, _compact_cache, _topo_cache, _ply_cache, train_aug=True, proto_data=_proto_data)
val_dataset = TriModalDataset(
    df_val, config, _compact_cache, _topo_cache, _ply_cache, train_aug=False, proto_data=_proto_data)

collate_fn = make_collate_fn(tokenizer, config.TEXT_MAX_LENGTH, use_task_prompt)

train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True,
    collate_fn=collate_fn, persistent_workers=(config.NUM_WORKERS > 0))
val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=True,
    collate_fn=collate_fn, persistent_workers=(config.NUM_WORKERS > 0))

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')


In [ ]:
# Cell: Create Model + Optimizer

model = Baseline(text_encoder, config).to(device)

n_text = sum(p.numel() for p in model.text_encoder.parameters()) / 1e6
n_brep = sum(p.numel() for p in model.brep_encoder.parameters()) / 1e6
n_pc = sum(p.numel() for p in model.pc_encoder.parameters()) / 1e6
print(f'Params: Text={n_text:.1f}M, BRep={n_brep:.1f}M, PC={n_pc:.1f}M, Total={n_text+n_brep+n_pc:.1f}M')

optimizer = optim.AdamW([
    {'params': model.text_encoder.parameters(), 'lr': config.LEARNING_RATE, 'weight_decay': config.TEXT_WEIGHT_DECAY},
    {'params': model.brep_encoder.parameters(), 'lr': config.LEARNING_RATE * config.BREP_LR_MULT, 'weight_decay': config.THREE_D_WEIGHT_DECAY},
    {'params': model.pc_encoder.parameters(), 'lr': config.LEARNING_RATE * config.PC_LR_MULT, 'weight_decay': config.THREE_D_WEIGHT_DECAY},
    {'params': [model.log_tau], 'lr': 1e-3, 'weight_decay': 0.0},
])

total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = len(train_loader) * config.WARMUP_EPOCHS

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.01 + 0.5 * 0.99 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
writer = SummaryWriter(os.path.join(config.LOCAL_SAVE_DIR, 'logs')) if config.ENABLE_TENSORBOARD else None

print(f'LR: text={config.LEARNING_RATE:.0e}, brep/pc={config.LEARNING_RATE*config.BREP_LR_MULT:.0e}')
print(f'WD: text={config.TEXT_WEIGHT_DECAY}, 3D={config.THREE_D_WEIGHT_DECAY}')
print(f'Steps: {total_steps} total, {warmup_steps} warmup')


In [ ]:
# Cell: Training Loop (12 epochs, text freeze at epoch 6)
import gc

metrics_history = []
best_recall = 0.0
text_frozen = False

for epoch in range(config.NUM_EPOCHS):
    print(f'\n{"=" * 60}')
    print(f'Epoch {epoch + 1}/{config.NUM_EPOCHS}')
    print(f'{"=" * 60}')

    # Freeze text encoder at epoch FREEZE_TEXT_AFTER_EPOCH
    if epoch >= config.FREEZE_TEXT_AFTER_EPOCH and not text_frozen:
        print(f'*** Freezing text encoder ***')
        for p in model.text_encoder.parameters():
            p.requires_grad = False
        model.text_encoder.eval()
        text_frozen = True
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
        print(f'    Trainable params: {n_trainable:.1f}M')

    train_loss, train_info = train_epoch(model, train_loader, optimizer, scheduler, config)
    val_loss, val_info = validate_epoch(model, val_loader, config)
    torch.cuda.empty_cache(); gc.collect()

    # Retrieval eval every epoch
    retrieval_metrics = evaluate_matryoshka(model, val_loader, config)

    # Print summary
    print(f'  Train: {train_loss:.4f}  Val: {val_loss:.4f}  tau: {model.tau.item():.4f}  frozen: {text_frozen}')

    # Train-val margin gap diagnostic
    for pair in ['tb', 'tp', 'bp']:
        tr = train_info.get(f'margin_{pair}_768d', 0)
        vl = val_info.get(f'margin_{pair}_768d', 0)
        gap = tr - vl
        tag = 'OK' if gap < 0.08 else 'OVERFIT' if gap > 0.15 else 'WATCH'
        print(f'  {pair} margin: train={tr:.4f} val={vl:.4f} gap={gap:+.4f} [{tag}]')

    print(f'\n  Retrieval @ {config.D_SHARED}d:')
    for d in ['text2brep', 'text2pc', 'brep2pc']:
        r1 = retrieval_metrics.get(f'{d}_recall@1_{config.D_SHARED}d', 0)
        r5 = retrieval_metrics.get(f'{d}_recall@5_{config.D_SHARED}d', 0)
        r10 = retrieval_metrics.get(f'{d}_recall@10_{config.D_SHARED}d', 0)
        print(f'    {d}: R@1={r1:.2f}%  R@5={r5:.2f}%  R@10={r10:.2f}%')

    # Logging
    epoch_metrics = {
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'tau': model.tau.item(), 'text_frozen': text_frozen,
        **{f'train_{k}': v for k, v in train_info.items()},
        **{f'val_{k}': v for k, v in val_info.items()},
        **retrieval_metrics,
    }
    metrics_history.append(epoch_metrics)

    if writer:
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('tau', model.tau.item(), epoch)
        for k, v in retrieval_metrics.items():
            writer.add_scalar(f'Retrieval/{k}', v, epoch)

    # Save best by text2brep R@1
    tb_r1 = retrieval_metrics.get(f'text2brep_recall@1_{config.D_SHARED}d', 0)
    if tb_r1 > best_recall:
        best_recall = tb_r1
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics,
                        config.LOCAL_SAVE_DIR, 'best.pth', light=False)
        sync_to_drive(
            os.path.join(config.LOCAL_SAVE_DIR, 'best.pth'),
            os.path.join(config.SAVE_DIR, 'best.pth'))
        print(f'  *** New best text2brep R@1: {best_recall:.2f}% ***')

    save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics,
                    config.LOCAL_SAVE_DIR, 'latest.pth', light=True)
    torch.cuda.empty_cache(); gc.collect()

if writer: writer.close()

# Final sync
print(f'\nDone. Best text2brep R@1: {best_recall:.2f}%')
import json as _json
with open(os.path.join(config.LOCAL_SAVE_DIR, 'metrics_history.json'), 'w') as f:
    _json.dump(metrics_history, f, indent=2)
sync_to_drive(os.path.join(config.LOCAL_SAVE_DIR, 'metrics_history.json'),
              os.path.join(config.SAVE_DIR, 'metrics_history.json'))
sync_to_drive(os.path.join(config.LOCAL_SAVE_DIR, 'latest.pth'),
              os.path.join(config.SAVE_DIR, 'latest.pth'))
print(f'Synced to {config.SAVE_DIR}')


In [ ]:
# Resume from epoch 7 checkpoint
import gc

ckpt_path = os.path.join(config.LOCAL_SAVE_DIR, 'best.pth')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(config.SAVE_DIR, 'best.pth')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
start_epoch = ckpt['epoch'] + 1  # resume from next epoch
print(f'Resumed from epoch {ckpt["epoch"]+1}, starting at epoch {start_epoch+1}')
print(f'Best text2brep R@1 so far: {ckpt["metrics"].get("text2brep_recall@1_768d", 0):.2f}%')

# Re-freeze text encoder (params were unfrozen by load_state_dict)
for p in model.text_encoder.parameters():
    p.requires_grad = False
model.text_encoder.eval()
print('Text encoder re-frozen.')

# Rebuild optimizer for only trainable params (skip text encoder)
optimizer = optim.AdamW([
    {'params': model.brep_encoder.parameters(), 'lr': config.LEARNING_RATE * config.BREP_LR_MULT, 'weight_decay': config.THREE_D_WEIGHT_DECAY},
    {'params': model.pc_encoder.parameters(), 'lr': config.LEARNING_RATE * config.PC_LR_MULT, 'weight_decay': config.THREE_D_WEIGHT_DECAY},
    {'params': [model.log_tau], 'lr': 1e-3, 'weight_decay': 0.0},
])

# Rebuild scheduler from where we left off
total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = len(train_loader) * config.WARMUP_EPOCHS
steps_done = len(train_loader) * start_epoch

def lr_lambda(step):
    actual_step = step + steps_done
    if actual_step < warmup_steps:
        return actual_step / max(1, warmup_steps)
    progress = (actual_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.01 + 0.5 * 0.99 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
writer = SummaryWriter(os.path.join(config.LOCAL_SAVE_DIR, 'logs')) if config.ENABLE_TENSORBOARD else None

best_recall = ckpt['metrics'].get('text2brep_recall@1_768d', 0)
metrics_history = []
text_frozen = True

# Run remaining epochs
for epoch in range(start_epoch, config.NUM_EPOCHS):
    print(f'\n{"=" * 60}')
    print(f'Epoch {epoch + 1}/{config.NUM_EPOCHS}')
    print(f'{"=" * 60}')

    train_loss, train_info = train_epoch(model, train_loader, optimizer, scheduler, config)
    val_loss, val_info = validate_epoch(model, val_loader, config)
    torch.cuda.empty_cache(); gc.collect()
    retrieval_metrics = evaluate_matryoshka(model, val_loader, config)

    print(f'  Train: {train_loss:.4f}  Val: {val_loss:.4f}  tau: {model.tau.item():.4f}  frozen: {text_frozen}')
    for pair in ['tb', 'tp', 'bp']:
        tr = train_info.get(f'margin_{pair}_768d', 0)
        vl = val_info.get(f'margin_{pair}_768d', 0)
        gap = tr - vl
        tag = 'OK' if gap < 0.08 else 'OVERFIT' if gap > 0.15 else 'WATCH'
        print(f'  {pair} margin: train={tr:.4f} val={vl:.4f} gap={gap:+.4f} [{tag}]')

    print(f'\n  Retrieval @ {config.D_SHARED}d:')
    for d in ['text2brep', 'text2pc', 'brep2pc']:
        r1 = retrieval_metrics.get(f'{d}_recall@1_{config.D_SHARED}d', 0)
        r5 = retrieval_metrics.get(f'{d}_recall@5_{config.D_SHARED}d', 0)
        r10 = retrieval_metrics.get(f'{d}_recall@10_{config.D_SHARED}d', 0)
        print(f'    {d}: R@1={r1:.2f}%  R@5={r5:.2f}%  R@10={r10:.2f}%')

    epoch_metrics = {
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'tau': model.tau.item(), 'text_frozen': text_frozen,
        **{f'train_{k}': v for k, v in train_info.items()},
        **{f'val_{k}': v for k, v in val_info.items()},
        **retrieval_metrics,
    }
    metrics_history.append(epoch_metrics)

    tb_r1 = retrieval_metrics.get(f'text2brep_recall@1_{config.D_SHARED}d', 0)
    if tb_r1 > best_recall:
        best_recall = tb_r1
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics,
                        config.LOCAL_SAVE_DIR, 'best.pth', light=False)
        sync_to_drive(os.path.join(config.LOCAL_SAVE_DIR, 'best.pth'),
                      os.path.join(config.SAVE_DIR, 'best.pth'))
        print(f'  *** New best text2brep R@1: {best_recall:.2f}% ***')

    save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics,
                    config.LOCAL_SAVE_DIR, 'latest.pth', light=True)
    torch.cuda.empty_cache(); gc.collect()

if writer: writer.close()
print(f'\nDone. Best text2brep R@1: {best_recall:.2f}%')
sync_to_drive(os.path.join(config.LOCAL_SAVE_DIR, 'latest.pth'),
              os.path.join(config.SAVE_DIR, 'latest.pth'))


In [ ]:
# Cell: Final Evaluation

for d in [config.LOCAL_SAVE_DIR, config.SAVE_DIR]:
    p = os.path.join(d, 'best.pth')
    if os.path.exists(p):
        ckpt = torch.load(p, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'Loaded best from epoch {ckpt["epoch"]+1} at {p}')
        break

final = evaluate_matryoshka(model, val_loader, config)

print(f'\n{"=" * 80}')
print('FINAL MATRYOSHKA RETRIEVAL MATRIX')
print(f'{"=" * 80}')

dirs = ['text2brep', 'brep2text', 'text2pc', 'pc2text', 'brep2pc', 'pc2brep']
for metric in ['recall@1', 'recall@5', 'recall@10']:
    print(f'\n{metric:>12}', end='')
    for dim in config.MATRYOSHKA_DIMS:
        print(f'{dim}d'.rjust(10), end='')
    print()
    print('-' * 52)
    for d in dirs:
        print(f'{d:>12}', end='')
        for dim in config.MATRYOSHKA_DIMS:
            v = final.get(f'{d}_{metric}_{dim}d', 0)
            print(f'{v:>9.2f}%', end='')
        print()

with open(os.path.join(config.LOCAL_SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump({'final': final, 'history': metrics_history}, f, indent=2)
sync_to_drive(os.path.join(config.LOCAL_SAVE_DIR, 'final_results.json'),
              os.path.join(config.SAVE_DIR, 'final_results.json'))
print(f'\nSaved to {config.SAVE_DIR}')


In [ ]:
# Cell: Export BRep anchors from the trained config-C model (for Phase 2/3 sketch/render + inference)
# Run AFTER training. Encodes every model's BRep embedding with the V-augmented encoder and saves
# brep_anchors.pt + split_info.pt to the config-C dir, so the sketch/render encoders align against
# the V-improved BRep space. (Not needed for the Phase-1 txt/pc->brep numbers, which Cell 'Final
# Evaluation' already produces.)
os.makedirs(config.SAVE_DIR, exist_ok=True)
for _d in [config.LOCAL_SAVE_DIR, config.SAVE_DIR]:
    _p = os.path.join(_d, 'best.pth')
    if os.path.exists(_p):
        _ck = torch.load(_p, map_location=device, weights_only=False)
        model.load_state_dict(_ck['model_state_dict']); print(f'Loaded best (epoch {_ck["epoch"]+1})'); del _ck; break
model.eval()

df_all = pd.concat([df_train, df_val]).reset_index(drop=True)
anchor_ds = TriModalDataset(df_all, config, _compact_cache, _topo_cache, _ply_cache,
                            train_aug=False, proto_data=_proto_data)
anchor_loader = DataLoader(anchor_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                           num_workers=config.NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
brep_anchors = {}
with torch.no_grad():
    for batch in tqdm(anchor_loader, desc='BRep anchors'):
        z = model.brep_encoder(
            batch['face_features'].to(device), batch['face_centroids'].to(device),
            batch['edge_to_faces'].to(device), batch['face_mask'].to(device),
            batch['face_normals'].to(device),
            proto_ids=batch['proto_ids'].to(device), proto_desc=batch['proto_desc'].to(device)).cpu()
        for _uid, _emb in zip(batch['uids'], z):
            brep_anchors[_uid] = _emb
torch.save(brep_anchors, os.path.join(config.SAVE_DIR, 'brep_anchors.pt'))
split_info = {'train_uids': list(train_dataset.samples['uid'].astype(str).str.strip()),
              'val_uids': list(val_dataset.samples['uid'].astype(str).str.strip())}
torch.save(split_info, os.path.join(config.SAVE_DIR, 'split_info.pt'))
print(f'Saved {len(brep_anchors)} BRep anchors + split_info to {config.SAVE_DIR}')
print('Phase 2/3: point the sketch/render notebook ANCHOR_DIR at this dir for the config-C ablation.')
